[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/17.%20The%20Container%20Reshuffling%20%28Remarshalling%29%20Problem/P17-Tier-3_executed.ipynb)

---


# 17. The Container Reshuffling (Remarshalling) Problem

## Problem, approach, and scope

A container is difficult to retrieve when other containers are stored above it. This notebook uses a Genetic Algorithm (GA) to search for stack assignments that reduce the number of relocations required by a specified retrieval simulation. The GA is a stochastic search method: it can find useful solutions, but it does not prove that a solution is globally optimal.

The input is a set of containers with target status and retrieval priority, together with the number of stacks and their height limit. A chromosome assigns each container to a stack. The output includes the best assignment found, its simulated relocation count and fitness, and statistics describing the evolutionary search.

### Model assumptions

- Containers are placed in the order in which they appear in `containers`. Within each stack, this order defines the vertical order from bottom to top. The chromosome chooses a stack for each container; it does not choose a separate within-stack order.
- Target containers are considered in ascending priority order, with the input order breaking ties. A target is retrieved when it is exposed. A blocking container is relocated to the nearest stack with available capacity, where stack IDs are treated as positions on a one-dimensional line.
- A relocation is counted each time a blocking container is moved. If the simulation cannot find a destination stack for a blocking container, it adds a penalty of 100 and stops processing that target. This penalty distinguishes a failed simulated retrieval from an ordinary relocation count; it is not a physical move.
- Container weight, destination, and arrival time are stored as data but do not affect this model's fitness. The model does not include crane travel time, safety constraints, or the cost of retrieving target containers.

The search follows the GA sequence used in this notebook: initialize a population, evaluate fitness, select parents by tournament selection, apply single-point crossover and mutation, repair stack-capacity violations, retain the best solution through elitism, and repeat until the generation limit or the convergence stopping rule is reached.

## Genetic Algorithm components

The example uses a population size of 50, crossover rate of 0.8, mutation rate of 0.15, and tournament size of 3. The run is limited to 100 generations and stops early if the best fitness does not improve for 15 consecutive generations. These are the settings for this example, not universal recommendations. No random seed is set, so separate runs can produce different search histories and solutions.

## Representation and fitness model

A chromosome is a vector of stack assignments. If there are $n$ containers and $m$ stacks, gene $g_i$ identifies the stack assigned to container $i$:

$$
g = (g_1, g_2, \ldots, g_n), \qquad g_i \in \{0, 1, \ldots, m-1\}.
$$

The stack-height constraint is that no stack receives more than $H$ containers, where $H$ is `max_height`. The code represents $g$ with `Chromosome.genes`; each gene is an integer stack ID. Crossover and mutation can temporarily violate the height constraint, so `repair_chromosome` restores feasibility before offspring are evaluated.

For a chromosome $g$, the retrieval simulation returns a relocation score $R(g)$. It counts ordinary relocations and adds the stated penalty when a target cannot be exposed under the greedy relocation rule. Fitness is

$$
F(g) = \frac{1000}{1 + R(g)}.
$$

Here, $F(g)$ is maximized by the GA; maximizing it is equivalent to minimizing $R(g)$. A score of 1000 corresponds to $R(g)=0$. The code computes this score in `evaluate_fitness`, which returns both fitness and the relocation score. Because the score comes from a specified greedy simulation, it evaluates assignments under that simulation rather than solving every possible relocation sequence optimally.

For a population of $P$ chromosomes, the algorithm repeatedly selects parents, creates and repairs offspring, and keeps an elite copy of the best chromosome. Tournament selection samples up to three individuals and chooses the one with highest fitness. Single-point crossover exchanges the gene suffixes after a randomly selected position. Mutation independently gives each gene a probability of 0.15 of receiving a randomly selected stack ID. These operators explore different assignments; repair enforces the stack-height constraint. The search is stochastic and does not guarantee a global optimum.

In [ ]:
# These libraries support the GA, its container data structures, and its plots.
import random
import time
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully for the container reshuffling Genetic Algorithm.")

In [ ]:
@dataclass
class Container:
    """Container attributes used by the example and retrieval simulation."""

    id: int
    weight: float  # Weight in tons; not used by the fitness model.
    destination: str  # Final destination; not used by the fitness model.
    priority: int  # Lower values are retrieved first.
    arrival_time: int  # Stored as input data; not used by the fitness model.
    is_target: bool = False
    blocking_count: int = 0  # Stored as input data; not used by the fitness model.


@dataclass
class Stack:
    """A stack whose last container in the list is its top container."""

    id: int
    max_height: int
    containers: List[Container] = field(default_factory=list)

    def add_container(self, container: Container) -> bool:
        """Add a container only when the stack has remaining capacity."""
        if len(self.containers) < self.max_height:
            self.containers.append(container)
            return True
        return False

    def remove_top_container(self) -> Optional[Container]:
        """Remove and return the exposed container, if the stack is nonempty."""
        if self.containers:
            return self.containers.pop()
        return None

    def get_top_container(self) -> Optional[Container]:
        """Return the exposed container without changing the stack."""
        if self.containers:
            return self.containers[-1]
        return None

    def is_full(self) -> bool:
        """Report whether the stack has reached its height limit."""
        return len(self.containers) >= self.max_height

    def get_available_space(self) -> int:
        """Return the number of unoccupied positions in the stack."""
        return self.max_height - len(self.containers)

    def copy(self) -> "Stack":
        """Copy the stack list while retaining the container records."""
        return Stack(self.id, self.max_height, self.containers.copy())


@dataclass
class Chromosome:
    """A chromosome encodes one stack assignment for every container."""

    genes: List[int]
    fitness: float = 0.0
    reshuffles: int = 0

    def copy(self) -> "Chromosome":
        """Copy the gene list and its evaluated scores."""
        return Chromosome(self.genes.copy(), self.fitness, self.reshuffles)


print("Container, Stack, and Chromosome data classes are ready.")

In [ ]:
class ContainerReshufflingGA:
    """Search stack assignments with a Genetic Algorithm."""

    def __init__(
        self,
        num_stacks: int,
        max_height: int,
        containers: List[Container],
        population_size: int = 50,
        crossover_rate: float = 0.8,
        mutation_rate: float = 0.15,
        tournament_size: int = 3,
    ):
        if num_stacks < 1 or max_height < 1:
            raise ValueError("The number of stacks and maximum height must be positive.")
        if not containers:
            raise ValueError("At least one container is required.")
        if len({container.id for container in containers}) != len(containers):
            raise ValueError("Container IDs must be unique.")
        if len(containers) > num_stacks * max_height:
            raise ValueError("The total stack capacity must accommodate every container.")
        if population_size < 1 or tournament_size < 1:
            raise ValueError("Population size and tournament size must be positive.")
        if not 0.0 <= crossover_rate <= 1.0:
            raise ValueError("Crossover rate must be between 0 and 1.")
        if not 0.0 <= mutation_rate <= 1.0:
            raise ValueError("Mutation rate must be between 0 and 1.")

        self.num_stacks = num_stacks
        self.max_height = max_height
        self.containers = containers
        self.target_containers = [container for container in containers if container.is_target]

        self.population_size = population_size
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.tournament_size = tournament_size

        self.population: List[Chromosome] = []
        self.best_solution: Optional[Chromosome] = None
        self.fitness_history: List[float] = []
        self.diversity_history: List[float] = []
        self.convergence_generation: Optional[int] = None
        self.generation_count = 0
        self.total_evaluations = 0

    def initialize_population(self, initial_assignment: Dict[int, int]) -> List[Chromosome]:
        """Include a balanced assignment and generate the remaining individuals."""
        print(f"Initializing a population of {self.population_size} individuals...")

        if set(initial_assignment) != {container.id for container in self.containers}:
            raise ValueError("The initial assignment must contain one entry for every container ID.")

        initial_genes = [initial_assignment[container.id] for container in self.containers]
        if any(not 0 <= stack_id < self.num_stacks for stack_id in initial_genes):
            raise ValueError("Every initial stack assignment must be a valid stack ID.")
        if any(initial_genes.count(stack_id) > self.max_height for stack_id in range(self.num_stacks)):
            raise ValueError("The initial assignment exceeds a stack's height limit.")

        population = [Chromosome(initial_genes)]
        for _ in range(self.population_size - 1):
            population.append(Chromosome(self._generate_random_chromosome()))
        return population

    def _generate_random_chromosome(self) -> List[int]:
        """Assign containers randomly while respecting each stack's capacity."""
        genes = []
        stack_loads = [0] * self.num_stacks

        for _container in self.containers:
            available_stacks = [
                stack_id
                for stack_id in range(self.num_stacks)
                if stack_loads[stack_id] < self.max_height
            ]
            chosen_stack = random.choice(available_stacks)
            genes.append(chosen_stack)
            stack_loads[chosen_stack] += 1

        return genes

    def evaluate_fitness(self, chromosome: Chromosome) -> Tuple[float, int]:
        """Simulate target retrieval and return fitness and its relocation score."""
        if len(chromosome.genes) != len(self.containers):
            raise ValueError("A chromosome must have one gene per container.")
        if any(not 0 <= gene < self.num_stacks for gene in chromosome.genes):
            raise ValueError("Every chromosome gene must be a valid stack ID.")

        temp_stacks = [Stack(stack_id, self.max_height) for stack_id in range(self.num_stacks)]
        for container, stack_id in zip(self.containers, chromosome.genes):
            if not temp_stacks[stack_id].add_container(container):
                raise ValueError("The chromosome exceeds a stack's height limit; repair it before evaluation.")

        reshuffles = 0
        retrieval_order = sorted(self.target_containers, key=lambda container: container.priority)

        for target_container in retrieval_order:
            target_stack = next(
                (
                    stack
                    for stack in temp_stacks
                    if any(container.id == target_container.id for container in stack.containers)
                ),
                None,
            )

            if target_stack is None:
                continue

            while target_stack.get_top_container() is not target_container:
                top_container = target_stack.get_top_container()
                if top_container is None:
                    reshuffles += 100
                    break

                destination_stack_id = self._find_nearest_available_stack(
                    target_stack.id, temp_stacks
                )
                if destination_stack_id is None:
                    # The penalty records a failed simulated retrieval, not a physical move.
                    reshuffles += 100
                    break

                moved_container = target_stack.remove_top_container()
                if moved_container is None or not temp_stacks[destination_stack_id].add_container(
                    moved_container
                ):
                    raise RuntimeError("The relocation simulation encountered an invalid stack move.")
                reshuffles += 1

            else:
                # The target must be removed even when it was exposed from the start.
                target_stack.remove_top_container()

        fitness = 1000.0 / (1 + reshuffles)
        return fitness, reshuffles

    def _find_nearest_available_stack(
        self, current_stack_id: int, stacks: List[Stack]
    ) -> Optional[int]:
        """Choose the closest non-full stack by absolute difference in stack IDs."""
        available_stack_ids = [
            stack.id
            for stack in stacks
            if stack.id != current_stack_id and not stack.is_full()
        ]
        if not available_stack_ids:
            return None
        return min(available_stack_ids, key=lambda stack_id: abs(stack_id - current_stack_id))

    def tournament_selection(self, population: List[Chromosome]) -> Chromosome:
        """Select the fittest individual from a randomly sampled tournament."""
        tournament = random.sample(population, min(self.tournament_size, len(population)))
        return max(tournament, key=lambda chromosome: chromosome.fitness)

    def crossover(
        self, parent1: Chromosome, parent2: Chromosome
    ) -> Tuple[Chromosome, Chromosome]:
        """Apply single-point crossover, or return parent copies when skipped."""
        if random.random() > self.crossover_rate or len(parent1.genes) < 2:
            return parent1.copy(), parent2.copy()

        crossover_point = random.randint(1, len(parent1.genes) - 1)
        offspring1_genes = parent1.genes[:crossover_point] + parent2.genes[crossover_point:]
        offspring2_genes = parent2.genes[:crossover_point] + parent1.genes[crossover_point:]
        return Chromosome(offspring1_genes), Chromosome(offspring2_genes)

    def mutate(self, chromosome: Chromosome) -> Chromosome:
        """Randomly reassign individual genes to maintain search variation."""
        mutated = chromosome.copy()
        for gene_index in range(len(mutated.genes)):
            if random.random() < self.mutation_rate:
                mutated.genes[gene_index] = random.randint(0, self.num_stacks - 1)
        return mutated

    def repair_chromosome(self, chromosome: Chromosome) -> Chromosome:
        """Move excess assignments to under-capacity stacks after variation."""
        repaired = chromosome.copy()
        if len(repaired.genes) != len(self.containers):
            raise ValueError("A chromosome must have one gene per container.")
        if any(not 0 <= gene < self.num_stacks for gene in repaired.genes):
            raise ValueError("Every chromosome gene must be a valid stack ID.")

        stack_counts = [0] * self.num_stacks
        for gene in repaired.genes:
            stack_counts[gene] += 1

        for gene_index, stack_id in enumerate(repaired.genes):
            if stack_counts[stack_id] > self.max_height:
                underloaded_stacks = [
                    stack_index
                    for stack_index in range(self.num_stacks)
                    if stack_counts[stack_index] < self.max_height
                ]
                if not underloaded_stacks:
                    raise RuntimeError("No under-capacity stack is available during chromosome repair.")

                new_stack_id = underloaded_stacks[0]
                stack_counts[stack_id] -= 1
                stack_counts[new_stack_id] += 1
                repaired.genes[gene_index] = new_stack_id

        return repaired

    def calculate_diversity(self, population: List[Chromosome]) -> float:
        """Return mean pairwise Hamming distance between chromosomes."""
        if len(population) < 2:
            return 0.0

        total_distance = 0
        comparisons = 0
        for first_index in range(len(population)):
            for second_index in range(first_index + 1, len(population)):
                distance = sum(
                    first_gene != second_gene
                    for first_gene, second_gene in zip(
                        population[first_index].genes,
                        population[second_index].genes,
                    )
                )
                total_distance += distance
                comparisons += 1

        return total_distance / comparisons

    def evolve(
        self, max_generations: int = 100, convergence_threshold: int = 10
    ) -> Dict:
        """Run population initialization and the GA generation loop."""
        if max_generations < 1:
            raise ValueError("The maximum number of generations must be positive.")
        if convergence_threshold < 1:
            raise ValueError("The convergence threshold must be positive.")

        print(f"Starting GA evolution with at most {max_generations} generations...")
        start_time = time.time()

        # Reset run-specific statistics so repeated calls start with fresh histories.
        self.fitness_history = []
        self.diversity_history = []
        self.convergence_generation = None
        self.generation_count = 0
        self.total_evaluations = 0
        self.best_solution = None

        self.population = self.initialize_population(self._get_initial_assignment())
        for chromosome in self.population:
            chromosome.fitness, chromosome.reshuffles = self.evaluate_fitness(chromosome)
            self.total_evaluations += 1

        self.best_solution = max(self.population, key=lambda chromosome: chromosome.fitness).copy()
        generations_without_improvement = 0
        generations_run = 0

        for generation in range(max_generations):
            self.generation_count = generation
            new_population = [self.best_solution.copy()]

            while len(new_population) < self.population_size:
                parent1 = self.tournament_selection(self.population)
                parent2 = self.tournament_selection(self.population)
                offspring1, offspring2 = self.crossover(parent1, parent2)
                offspring1 = self.repair_chromosome(self.mutate(offspring1))
                offspring2 = self.repair_chromosome(self.mutate(offspring2))
                new_population.extend([offspring1, offspring2])

            self.population = new_population[:self.population_size]
            for chromosome in self.population:
                chromosome.fitness, chromosome.reshuffles = self.evaluate_fitness(chromosome)
                self.total_evaluations += 1

            current_best = max(self.population, key=lambda chromosome: chromosome.fitness)
            if current_best.fitness > self.best_solution.fitness:
                self.best_solution = current_best.copy()
                generations_without_improvement = 0
            else:
                generations_without_improvement += 1

            average_fitness = float(np.mean([chromosome.fitness for chromosome in self.population]))
            diversity = self.calculate_diversity(self.population)
            self.fitness_history.append(average_fitness)
            self.diversity_history.append(diversity)
            generations_run = generation + 1

            if generations_without_improvement >= convergence_threshold:
                self.convergence_generation = generation
                print(f"Convergence stopping rule reached at generation {generation}.")
                break

            if generation % 10 == 0:
                print(
                    f"Generation {generation}: best fitness = {self.best_solution.fitness:.2f}, "
                    f"average fitness = {average_fitness:.2f}, diversity = {diversity:.2f}"
                )

        solve_time = time.time() - start_time
        return {
            "best_fitness": self.best_solution.fitness,
            "best_reshuffles": self.best_solution.reshuffles,
            "best_solution": self.best_solution.genes.copy(),
            "generations": generations_run,
            "solve_time": solve_time,
            "total_evaluations": self.total_evaluations,
            "convergence_generation": self.convergence_generation,
            "fitness_history": self.fitness_history.copy(),
            "diversity_history": self.diversity_history.copy(),
        }

    def _get_initial_assignment(self) -> Dict[int, int]:
        """Create a balanced, capacity-feasible starting assignment."""
        assignment = {}
        stack_loads = [0] * self.num_stacks

        for container in self.containers:
            available_stacks = [
                stack_id
                for stack_id in range(self.num_stacks)
                if stack_loads[stack_id] < self.max_height
            ]
            chosen_stack = min(available_stacks, key=lambda stack_id: stack_loads[stack_id])
            assignment[container.id] = chosen_stack
            stack_loads[chosen_stack] += 1

        return assignment


print("ContainerReshufflingGA is ready.")

## Example data and GA configuration

The container records below define the example instance. There are eight containers, four stacks, and a maximum stack height of three, so the total capacity is sufficient for the assignment. Two containers are retrieval targets. Their priority values determine the simulation order; the other stored attributes are included in the records but do not affect the fitness calculation.

In [ ]:
print("Creating the container reshuffling example...")

containers = [
    Container(id=1, weight=20.5, destination="NYC", priority=2, arrival_time=0),
    Container(id=2, weight=18.2, destination="LAX", priority=1, arrival_time=0, is_target=True),
    Container(id=3, weight=22.1, destination="CHI", priority=3, arrival_time=0),
    Container(id=4, weight=19.8, destination="MIA", priority=2, arrival_time=0),
    Container(id=5, weight=21.3, destination="SEA", priority=1, arrival_time=0, is_target=True),
    Container(id=6, weight=17.9, destination="BOS", priority=2, arrival_time=0),
    Container(id=7, weight=23.4, destination="DEN", priority=3, arrival_time=0),
    Container(id=8, weight=20.1, destination="ATL", priority=2, arrival_time=0),
]

num_stacks = 4
max_height = 3
population_size = 50
crossover_rate = 0.8
mutation_rate = 0.15
tournament_size = 3

ga = ContainerReshufflingGA(
    num_stacks=num_stacks,
    max_height=max_height,
    containers=containers,
    population_size=population_size,
    crossover_rate=crossover_rate,
    mutation_rate=mutation_rate,
    tournament_size=tournament_size,
)

print("GA configuration:")
print(f"- Instance: {num_stacks} stacks, maximum height {max_height}, {len(containers)} containers")
print(f"- Target containers: {len(ga.target_containers)}")
print(f"- Population size: {population_size}")
print(f"- Crossover rate: {crossover_rate}")
print(f"- Mutation rate: {mutation_rate}")
print(f"- Tournament size: {tournament_size}")

## Run the search and inspect the assignment

The generation limit bounds the computational effort. The convergence threshold is a separate stopping rule: the run ends early if the best fitness fails to improve for the specified number of consecutive generations. The reported assignment is ordered in the same way as `containers`, so each printed stack ID corresponds to the container on that line.

In [ ]:
print("\n=== Running the Genetic Algorithm ===")
ga_result = ga.evolve(max_generations=100, convergence_threshold=15)

print("\n=== Search results ===")
print(f"Best fitness: {ga_result['best_fitness']:.2f}")
print(f"Relocation score: {ga_result['best_reshuffles']}")
print(f"Generations run: {ga_result['generations']}")
print(f"Solve time: {ga_result['solve_time']:.2f} seconds")
print(f"Fitness evaluations: {ga_result['total_evaluations']}")
print(f"Convergence generation: {ga_result['convergence_generation']}")

print("\nBest container-to-stack assignment:")
for container, stack_id in zip(containers, ga_result["best_solution"]):
    status = "Target" if container.is_target else "Non-target"
    print(
        f"  Container {container.id} -> Stack {stack_id} "
        f"({status}, priority {container.priority})"
    )

## Evolution and population diagnostics

Average fitness and population diversity describe different aspects of the search. Average fitness summarizes the current population's evaluated quality. Diversity is the mean pairwise Hamming distance: for two chromosomes, Hamming distance counts the positions assigned to different stacks. A falling diversity value can indicate that assignments are becoming more alike, but it does not by itself prove that the search has found a global optimum.

The final-population histograms describe the population at termination. They are not a comparison against another algorithm or a guarantee of solution quality.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

generations = list(range(len(ga_result["fitness_history"])))
ax1.plot(
    generations,
    ga_result["fitness_history"],
    color="blue",
    linewidth=2,
    label="Average population fitness",
)
ax1.axhline(
    y=ga_result["best_fitness"],
    color="red",
    linestyle="--",
    label=f"Best fitness: {ga_result['best_fitness']:.2f}",
)
ax1.set_xlabel("Generation")
ax1.set_ylabel("Fitness (higher is better)")
ax1.set_title("Fitness During the Search")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(
    generations,
    ga_result["diversity_history"],
    color="green",
    linewidth=2,
    label="Mean pairwise Hamming distance",
)
ax2.set_xlabel("Generation")
ax2.set_ylabel("Diversity (gene positions)")
ax2.set_title("Population Diversity During the Search")
ax2.legend()
ax2.grid(True, alpha=0.3)

final_fitness = [chromosome.fitness for chromosome in ga.population]
ax3.hist(final_fitness, bins=20, alpha=0.7, color="skyblue", edgecolor="black")
ax3.axvline(
    x=ga_result["best_fitness"],
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Best: {ga_result['best_fitness']:.2f}",
)
ax3.axvline(
    x=np.mean(final_fitness),
    color="orange",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {np.mean(final_fitness):.2f}",
)
ax3.set_xlabel("Fitness (higher is better)")
ax3.set_ylabel("Number of individuals")
ax3.set_title("Final-Population Fitness Distribution")
ax3.legend()
ax3.grid(True, alpha=0.3)

final_reshuffles = [chromosome.reshuffles for chromosome in ga.population]
ax4.hist(final_reshuffles, bins=15, alpha=0.7, color="lightgreen", edgecolor="black")
ax4.axvline(
    x=ga_result["best_reshuffles"],
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Best score: {ga_result['best_reshuffles']}",
)
ax4.axvline(
    x=np.mean(final_reshuffles),
    color="orange",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {np.mean(final_reshuffles):.1f}",
)
ax4.set_xlabel("Relocation score (lower is better)")
ax4.set_ylabel("Number of individuals")
ax4.set_title("Final-Population Relocation-Score Distribution")
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Final-population statistics ===")
print(f"Fitness — mean: {np.mean(final_fitness):.2f}, standard deviation: {np.std(final_fitness):.2f}")
print(f"Fitness — minimum: {np.min(final_fitness):.2f}, maximum: {np.max(final_fitness):.2f}")
print(
    f"Relocation score — mean: {np.mean(final_reshuffles):.1f}, "
    f"standard deviation: {np.std(final_reshuffles):.2f}"
)
print(
    f"Relocation score — minimum: {np.min(final_reshuffles)}, "
    f"maximum: {np.max(final_reshuffles)}"
)

## Summary and interpretation

This notebook demonstrates a complete Genetic Algorithm for searching container-to-stack assignments. Chromosomes encode assignments, the fitness function evaluates them through a priority-based retrieval simulation, and tournament selection, single-point crossover, mutation, repair, and elitism drive the search. The fitness and diversity histories provide diagnostics for the run, while the final assignment and histograms show the resulting candidate and population.

Interpret the reported relocation score relative to the model assumptions: it is the score produced by the specified greedy relocation simulation, including its penalty for a target that cannot be exposed. It is not a proof of the minimum physically achievable number of moves. Results can vary between runs because the GA uses random choices and this example does not set a random seed. Multiple assignments may also have the same fitness.

The GA is useful when a flexible stochastic search is acceptable and a guaranteed optimum is not required. Its limitations include sensitivity to parameter settings, computation spent evaluating many candidates, and the absence of an optimality guarantee. This example does not include a parameter sensitivity study or a measured comparison with other solution methods, so no general performance improvement or runtime ratio can be inferred from it.

Natural extensions within the same modeling approach include larger test instances and additional operational constraints or costs. Such extensions require corresponding changes to the assignment representation or fitness simulation and should be interpreted against their explicitly stated assumptions.